In [0]:
from pyspark.sql.functions import (
    col, trim, upper, split, explode, regexp_extract, when, lit, length,
    expr, count, lower, substring, monotonically_increasing_id, concat, sha2
)

dallas_bronze = spark.table("workspace.default.bronze_dallas_inspections")
print(f"Bronze rows: {dallas_bronze.count()}")

In [0]:
# ============================================================================
# SILVER LAYER VALIDATION RULES (Dallas)
# ============================================================================

dallas_validated = (
    dallas_bronze
    # Rule 1: Restaurant name cannot be null
    .where(col("restaurant_name").isNotNull() & (length(trim(col("restaurant_name"))) > 0))
    # Rule 2: Inspection Date cannot be null
    .where(col("inspection_date").isNotNull())
    # Rule 3: Inspection Type cannot be null
    .where(col("inspection_type").isNotNull() & (length(trim(col("inspection_type"))) > 0))
    # Rule 4: Zip code cannot be null — truncate ZIP+4 to 5 digits
    .where(col("zip_code").isNotNull() & (length(trim(col("zip_code"))) >= 5))
    .withColumn("zip_code_clean", substring(trim(col("zip_code")), 1, 5))
    # Validate zip is 5 digits after truncation
    .where(col("zip_code_clean").rlike("^\\d{5}$"))
    # Rule 5: Violation score cannot exceed 100
    .where(col("inspection_score") <= 100)
    # Rule 11: Negative inspection scores are invalid
    .where(col("inspection_score") >= 0)
)

dropped = dallas_bronze.count() - dallas_validated.count()
print(f"Rows after validation: {dallas_validated.count()} (dropped {dropped})")

In [0]:
# ============================================================================
# UNPIVOT DALLAS VIOLATIONS
# Dallas has 25 violation slot groups (violation_description_N, points_N, detail_N, memo_N)
# We stack them into rows, dropping empty slots
# ============================================================================

stack_expr = "stack(25, " + ", ".join([
    f"'{i}', violation_description_{i}, violation_points_{i}, violation_detail_{i}, violation_memo_{i}"
    for i in range(1, 26)
]) + ") as (violation_slot, violation_description, violation_points, violation_detail, violation_memo)"

dallas_unpivoted = (
    dallas_validated
    .select(
        "restaurant_name", "inspection_type", "inspection_date", "inspection_score",
        "street_address", "zip_code_clean", "lat_long_location",
        "street_number", "street_name", "street_direction", "street_type", "street_unit",
        "inspection_month", "inspection_year",
        expr(stack_expr)
    )
    # Drop empty violation slots
    .where(col("violation_description").isNotNull() & (length(trim(col("violation_description"))) > 0))
    # Cast violation_slot from string to int
    .withColumn("violation_slot", col("violation_slot").cast("int"))
)

print(f"Total violation rows after unpivot: {dallas_unpivoted.count()}")

In [0]:
# ============================================================================
# ASSIGNMENT VALIDATION RULES (Dallas-specific)
# Rule 8: Score >= 90 cannot have more than 3 violations
# Rule 9: Result cannot be PASS if any violation contains Urgent/Critical
# ============================================================================
from pyspark.sql.window import Window

# Count violations per inspection (using restaurant_name + date as composite key)
from pyspark.sql.functions import count as spark_count, max as spark_max

violation_counts = (
    dallas_unpivoted
    .groupBy("restaurant_name", "inspection_date", "inspection_score")
    .agg(
        spark_count("*").alias("violation_count"),
        # Check if any violation contains Urgent/Critical keywords
        spark_max(
            when(col("violation_description").rlike("(?i)(urgent|critical)"), lit(True)).otherwise(lit(False))
        ).alias("has_urgent_critical")
    )
)

# Rule 8: Flag inspections with score >= 90 and more than 3 violations
rule8_violations = violation_counts.where(
    (col("inspection_score") >= 90) & (col("violation_count") > 3)
)
print(f"Rule 8 — Score >= 90 with > 3 violations: {rule8_violations.count()}")

# Rule 9: Flag inspections where score >= 90 but has Urgent/Critical violations
rule9_violations = violation_counts.where(
    (col("inspection_score") >= 90) & (col("has_urgent_critical") == True)
)
print(f"Rule 9 — Score >= 90 with Urgent/Critical: {rule9_violations.count()}")

# Create set of bad inspections to exclude
bad_inspections = (
    rule8_violations.select("restaurant_name", "inspection_date")
    .union(rule9_violations.select("restaurant_name", "inspection_date"))
    .distinct()
)
print(f"Total bad inspections to drop: {bad_inspections.count()}")

In [0]:
# ============================================================================
# Remove inspections that violate Rules 8 and 9
# Then deduplicate violations per inspection (Rule 7)
# ============================================================================

dallas_clean = (
    dallas_unpivoted
    .join(bad_inspections, on=["restaurant_name", "inspection_date"], how="left_anti")
)

print(f"Rows after Rules 8 & 9 filter: {dallas_clean.count()}")

# Rule 7: Deduplicate violations within the same inspection
dallas_deduped = dallas_clean.dropDuplicates(
    ["restaurant_name", "inspection_date", "violation_description"]
)

print(f"After dedup: {dallas_deduped.count()}")
print(f"Duplicates removed: {dallas_clean.count() - dallas_deduped.count()}")

In [0]:
# ============================================================================
# BUILD DALLAS SILVER TABLE
# Parse violation codes from description prefix (e.g., "*31 Description" -> code 31)
# Parse lat/long from combined field (e.g., "(32.793, -96.747)" -> lat, long)
# Standardize schema to align with Chicago Silver
# ============================================================================

dallas_silver = (
    dallas_deduped
    .withColumn("violation_code", regexp_extract(col("violation_description"), r"\*?(\d+)", 1))
    # Parse lat/long from combined field
    .withColumn("latitude", 
        when(col("lat_long_location").isNotNull(),
             regexp_extract(col("lat_long_location"), r"\(([^,]+)", 1).cast("double"))
        .otherwise(lit(None).cast("double")))
    .withColumn("longitude",
        when(col("lat_long_location").isNotNull(),
             regexp_extract(col("lat_long_location"), r",\s*([^)]+)", 1).cast("double"))
        .otherwise(lit(None).cast("double")))
    # Generate a synthetic inspection_id for Dallas (has no native ID)
    .withColumn("inspection_id",
        concat(lit("DAL-"), sha2(concat(col("restaurant_name"), col("inspection_date").cast("string")), 256).substr(1, 12)))
    # Derive result text from score
    .withColumn("results",
        when(col("inspection_score") >= 90, "Pass")
        .when(col("inspection_score") >= 80, "Pass w/ Conditions")
        .when(col("inspection_score") >= 70, "Fail")
        .when(col("inspection_score") > 0, "Fail")
        .otherwise("No Entry"))
    # Derive severity from points and keywords
    .withColumn("violation_severity",
        when(
            (col("violation_points") == 3) | col("violation_description").rlike("(?i)(urgent|critical)"), "Critical"
        ).when(col("violation_points") == 2, "Serious")
        .otherwise("Minor"))
    .select(
        lit("Dallas").alias("source_city"),
        col("inspection_id"),
        col("inspection_date"),
        upper(trim(col("restaurant_name"))).alias("dba_name"),
        lit(None).cast("string").alias("aka_name"),           # Dallas has no AKA
        lit(None).cast("string").alias("license_number"),     # Dallas has no license
        lit("Restaurant").alias("facility_type"),              # Default for Dallas
        lit(None).cast("string").alias("risk_level"),         # Dallas has no risk
        upper(trim(col("street_address"))).alias("address"),
        lit("DALLAS").alias("city"),
        lit("TX").alias("state"),
        col("zip_code_clean").alias("zip_code"),
        col("latitude"),
        col("longitude"),
        trim(col("inspection_type")).alias("inspection_type"),
        col("results"),
        col("inspection_score"),
        col("violation_code"),
        trim(col("violation_description")).alias("violation_description"),
        trim(col("violation_detail")).alias("violation_detail"),
        col("violation_points"),
        col("violation_slot"),
        col("violation_severity")
    )
)

print(f"Dallas Silver rows: {dallas_silver.count()}")
dallas_silver.printSchema()

In [0]:
dallas_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.silver_dallas_inspections")

print(f"Silver table written: {spark.table('workspace.default.silver_dallas_inspections').count()} rows")

In [0]:
silver = spark.table("workspace.default.silver_dallas_inspections")

print("=== DALLAS SILVER VALIDATION SUMMARY ===")
print(f"Total rows: {silver.count()}")
print(f"Unique inspections: {silver.select('inspection_id').distinct().count()}")
print(f"Unique violation codes: {silver.select('violation_code').distinct().count()}")
print(f"Date range: ", end="")
display(silver.selectExpr("min(inspection_date) as earliest", "max(inspection_date) as latest"))

print("\n=== NULL CHECK ON REQUIRED FIELDS ===")
for c in ["dba_name", "inspection_date", "inspection_type", "zip_code"]:
    nulls = silver.where(col(c).isNull()).count()
    print(f"  {c}: {nulls} nulls {'✓' if nulls == 0 else '✗ FAIL'}")

print("\n=== SCORE DISTRIBUTION ===")
display(silver.groupBy("results", "inspection_score").count().orderBy("inspection_score"))

print("\n=== SEVERITY DISTRIBUTION ===")
display(silver.groupBy("violation_severity").count().orderBy(col("count").desc()))

print("\n=== RULE 8 VERIFICATION ===")
rule8_check = (
    silver
    .groupBy("inspection_id", "inspection_score")
    .count()
    .where((col("inspection_score") >= 90) & (col("count") > 3))
)
print(f"Violations of Rule 8 remaining: {rule8_check.count()} {'✓' if rule8_check.count() == 0 else '✗ FAIL'}")